# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Coder-bot1/FlyRank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Contract

**1. What one row means:**  
One row represents one content item for one client on one report date. For the Search Intelligence slice, I use March 2026 as the development and decision month.

**2. Tables used:**  
I use `fact_content_daily_performance` for daily search-performance and availability data, and `dim_content` for content attributes. The daily table is verified at `report_date × client_hash_id × content_hash_id` grain.

**3. Time window:**  
March 2026 is the development/decision window. April 2026 is used only as the future outcome window for the label. June 2026 is treated as a sealed final month and is not used to develop the label logic.

**4. What I predict/rank:**  
I predict future GSC clicks for each client-content pair in April 2026. This is a directional decision-support proxy for future search performance.

**5. What I deliberately exclude:**  
I exclude future April performance from the normal feature set because it is not knowable at the March decision moment. I also exclude client names, URLs, and raw/private queries.

In [1]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found. Add HF_TOKEN to Colab Secrets "
        "and enable notebook access."
    )

print("HF_TOKEN loaded successfully.")

HF_TOKEN loaded successfully.


In [2]:
from huggingface_hub import hf_hub_download

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

print(march_file)

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [3]:
import pandas as pd

march_df = pd.read_parquet(march_file)

march_df["report_date"] = pd.to_datetime(
    march_df["report_date"]
)

print("March rows:", len(march_df))
print(
    "Date range:",
    march_df["report_date"].min(),
    "to",
    march_df["report_date"].max()
)

March rows: 9841378
Date range: 2026-03-01 00:00:00 to 2026-03-31 00:00:00


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field contract

**Features — five maximum**

1. `gsc_impressions` — available at the March decision moment because it is observed before the April outcome window.
2. `gsc_clicks` — available at the March decision moment because March clicks are historical information when making the April prediction.
3. `gsc_avg_position` — available at the March decision moment because March search position is already observed.
4. `sessions_organic` — available at the March decision moment because March organic sessions are historical traffic information.
5. `scroll_events` — available at the March decision moment because March engagement events are already observed.

**Label**

`future_clicks` — the total GSC clicks observed for the same client-content pair during April 2026. This is the future outcome and is not used as an honest feature.

**Context**

`client_hash_id`, `content_hash_id`, and `report_date` identify the client, content, and observation date. They are used for grouping and joining, not as predictive features.

**Excluded**

April performance fields are deliberately excluded from the honest feature set because they are not available at the March decision moment. `gsc_clicks` and other March metrics are only used as historical features; future April values are not used except in the deliberate leakage experiment.

In [4]:
# Define the five honest features and the identifiers/context fields.

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "sessions_organic",
    "scroll_events"
]

context_cols = [
    "client_hash_id",
    "content_hash_id",
    "report_date"
]

print("Five features:")
for feature in feature_cols:
    print("-", feature)

print("\nContext fields:")
for field in context_cols:
    print("-", field)

Five features:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- sessions_organic
- scroll_events

Context fields:
- client_hash_id
- content_hash_id
- report_date


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Three verification queries

I use March 2026 as the mid-panel development month. The three checks below verify the observed grain, the slice size/date range, and GSC data availability. The availability check deliberately uses `IS TRUE` so unavailable data is not treated as zero performance.

In [5]:
import duckdb

con = duckdb.connect()

march_path = march_file

# QUERY 1 — Grain
grain_query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT
        CONCAT(
            CAST(report_date AS VARCHAR), '|',
            client_hash_id, '|',
            content_hash_id
        )
    ) AS distinct_grain_rows
FROM read_parquet('{march_path}')
"""

grain_result = con.execute(grain_query).df()

print("QUERY 1 — GRAIN")
display(grain_result)

# QUERY 2 — Row count and date span
count_date_query = f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM read_parquet('{march_path}')
"""

count_date_result = con.execute(count_date_query).df()

print("\nQUERY 2 — ROW COUNT AND DATE SPAN")
display(count_date_result)

# QUERY 3 — GSC availability using IS TRUE
availability_query = f"""
SELECT
    COUNT(*) AS available_gsc_rows
FROM read_parquet('{march_path}')
WHERE gsc_data_available IS TRUE
"""

availability_result = con.execute(availability_query).df()

print("\nQUERY 3 — GSC AVAILABILITY")
display(availability_result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

QUERY 1 — GRAIN


,total_rows,distinct_grain_rows
0,9841378,9841378



QUERY 2 — ROW COUNT AND DATE SPAN


,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31



QUERY 3 — GSC AVAILABILITY


,available_gsc_rows
0,3611061


### Five-feature frame

For the March decision snapshot, I aggregate daily observations to the `client_hash_id × content_hash_id` level. I use only rows where `gsc_data_available IS TRUE` for GSC-based features, so missing/unavailable GSC data is not interpreted as zero. The five features are historical March signals that would be knowable when making an April decision.

| Feature | Available when? |
|---|---|
| gsc_impressions | Knowable at the decision moment because March impressions are already observed. |
| gsc_clicks | Knowable at the decision moment because March clicks are historical information. |
| gsc_avg_position | Knowable at the decision moment because March search position is already observed. |
| sessions_organic | Knowable at the decision moment because March organic sessions are already observed. |
| scroll_events | Knowable at the decision moment because March engagement events are already observed. |

In [6]:
# Build the five-feature frame from March 2026.
# GSC-based features use only rows where GSC data is actually available.

march_available = march_df[
    march_df["gsc_data_available"].eq(True)
].copy()

feature_frame = (
    march_available
    .groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        gsc_impressions=("gsc_impressions", "sum"),
        gsc_clicks=("gsc_clicks", "sum"),
        gsc_avg_position=("gsc_avg_position", "mean"),
        sessions_organic=("sessions_organic", "sum"),
        scroll_events=("scroll_events", "sum")
    )
)

print("Feature frame shape:", feature_frame.shape)
print("Number of features:", len(feature_cols))

display(feature_frame.head())

Feature frame shape: (176738, 7)
Number of features: 5


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,sessions_organic,scroll_events
0,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1,0,9.000000,0.0,0.0
1,client_0797ff3a1fc9a6a5,content_04c67f3541177192,331,2,14.129210,0.0,0.0
2,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,33,0,9.225529,0.0,0.0
3,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,145,0,8.470926,0.0,0.0
4,client_0797ff3a1fc9a6a5,content_1207efddce873942,461,0,14.859827,0.0,0.0


### Deliberate leakage experiment

To demonstrate target leakage, I intentionally add the future label `future_clicks` as a model feature. This feature is unavailable at the March decision moment because it comes from April. I compare the resulting score with the honest model, then remove the leaked column and retain the honest result.

In [7]:
# Load April 2026, which is the future outcome window.

april_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

april_df = pd.read_parquet(april_file)

april_df["report_date"] = pd.to_datetime(april_df["report_date"])

# Future label: April GSC clicks.
# Only rows with actual GSC data availability are included.

april_available = april_df[
    april_df["gsc_data_available"].eq(True)
].copy()

future_label = (
    april_available
    .groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        future_clicks=("gsc_clicks", "sum")
    )
)

model_df = feature_frame.merge(
    future_label,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Model frame shape:", model_df.shape)

display(model_df.head())

Model frame shape: (158549, 8)


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,sessions_organic,scroll_events,future_clicks
0,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1,0,9.000000,0.0,0.0,0
1,client_0797ff3a1fc9a6a5,content_04c67f3541177192,331,2,14.129210,0.0,0.0,3
2,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,145,0,8.470926,0.0,0.0,0
3,client_0797ff3a1fc9a6a5,content_1207efddce873942,461,0,14.859827,0.0,0.0,0
4,client_0797ff3a1fc9a6a5,content_12890868e4cdac06,1,0,19.000000,0.0,0.0,0


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

# Exactly five honest features.
X = model_df[feature_cols]
y = model_df["future_clicks"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

honest_model = RandomForestRegressor(
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)

honest_model.fit(X_train, y_train)

honest_predictions = honest_model.predict(X_test)

honest_r2 = r2_score(
    y_test,
    honest_predictions
)

print("Honest R²:", honest_r2)

Honest R²: 0.46420242982914095


In [9]:
# DELIBERATE LEAK:
# future_clicks is literally the target, so this feature would not
# exist at the March decision moment.

model_df["leaky_future_clicks"] = model_df["future_clicks"]

leaky_features = feature_cols + ["leaky_future_clicks"]

X_leaky = model_df[leaky_features]
y_leaky = model_df["future_clicks"]

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leaky,
    y_leaky,
    test_size=0.20,
    random_state=42
)

leaky_model = RandomForestRegressor(
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)

leaky_model.fit(X_train_l, y_train_l)

leaky_predictions = leaky_model.predict(X_test_l)

leaky_r2 = r2_score(
    y_test_l,
    leaky_predictions
)

print("Honest R²:", honest_r2)
print("R² with deliberate leakage:", leaky_r2)
print("R² increase:", leaky_r2 - honest_r2)

Honest R²: 0.46420242982914095
R² with deliberate leakage: 0.580871275721196
R² increase: 0.11666884589205506


In [10]:
# Remove the deliberately leaked target-derived feature.

model_df = model_df.drop(
    columns=["leaky_future_clicks"]
)

print("Leaky feature removed.")
print("Remaining model features:")
print(feature_cols)

Leaky feature removed.
Remaining model features:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'sessions_organic', 'scroll_events']


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Limitation

One important limitation is that GSC and GA4 availability is uneven across the warehouse. In March 2026, only 3,611,061 of 9,841,378 rows had `gsc_data_available = TRUE`, so the Search Intelligence slice is not equally observed for every client-content-day. This means the resulting features and future-click label describe the observed/available GSC population and should be treated as directional decision-support signals rather than a complete view of all content performance.

In [11]:
# Quantify the availability limitation described above.

total_march_rows = len(march_df)

available_gsc_rows = int(
    march_df["gsc_data_available"].eq(True).sum()
)

availability_rate = (
    available_gsc_rows / total_march_rows
)

print("Total March rows:", total_march_rows)
print("Rows with GSC available:", available_gsc_rows)
print("GSC availability rate:", round(availability_rate, 4))

Total March rows: 9841378
Rows with GSC available: 3611061
GSC availability rate: 0.3669


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.